# Responses API + 结构化输出

用zenmux救活了这个任务

In [2]:
import os
from dotenv import load_dotenv
from rich import print

In [ ]:
load_dotenv("../.env/zenmux", override=True)

True

In [4]:
model_id = os.environ["OPENAI_RESPONSES_MODEL_ID"]
model_id

'openai/gpt-5.4-mini'

In [5]:
import os
from openai import OpenAI
from pydantic import BaseModel

client = OpenAI()


class Step(BaseModel):
    explanation: str
    output: str


class Solving(BaseModel):
    steps: list[Step]
    final_answer: str


## 使用chat/completions接口 + Firework AI接口 ✅

In [6]:
response = client.chat.completions.create(
    model=model_id,
    response_format={
        "type": "json_schema",
        "json_schema": {"name": "Solving", "schema": Solving.model_json_schema()},
    },
    messages=[
        {
            "role": "user",
            "content": "how can I solve 8x + 7y = -23, and 4x=12? Reply in JSON format.",
        }
    ],
)

BadRequestError: Error code: 400 - {'error': {'code': '400', 'type': 'invalid_params', 'message': "Invalid schema for response_format 'Solving': In context=(), 'additionalProperties' is required to be supplied and to be false."}}

In [33]:
print(response.choices[0].message.content)

{
"steps": [
{
"explanation": "First, solve the second equation for x since it only contains one variable",
"output": "4x = 12\nx = 12/4\nx = 3"
},
{
"explanation": "Substitute x = 3 into the first equation to solve for y",
"output": "8(3) + 7y = -23\n24 + 7y = -23"
},
{
"explanation": "Isolate the y term by subtracting 24 from both sides",
"output": "7y = -23 - 24\n7y = -47"
},
{
"explanation": "Solve for y by dividing both sides by 7",
"output": "y = -47/7\ny = -6.714... (or y = -47/7 as exact fraction)"
}
],
"final_answer": "x = 3, y = -47/7 (or approximately y = -6.71)"
}

## 使用responses接口

- responses.create() 提供 text参数，指定json_schema
- responses.parse() 提供text_foramt参数，指定Pydantic类

### 使用responses.create

In [18]:
Solving.model_json_schema()

{'$defs': {'Step': {'properties': {'explanation': {'title': 'Explanation',
     'type': 'string'},
    'output': {'title': 'Output', 'type': 'string'}},
   'required': ['explanation', 'output'],
   'title': 'Step',
   'type': 'object'}},
 'properties': {'steps': {'items': {'$ref': '#/$defs/Step'},
   'title': 'Steps',
   'type': 'array'},
  'final_answer': {'title': 'Final Answer', 'type': 'string'}},
 'required': ['steps', 'final_answer'],
 'title': 'Solving',
 'type': 'object'}

In [19]:
resp_c = client.responses.create(
    model=model_id,
    text={
        "format": {
            "type": "json_schema",
            "name": "Solving",
            "schema": {
                "$defs": {
                    "Step": {
                        "properties": {
                            "explanation": {"title": "Explanation", "type": "string"},
                            "output": {"title": "Output", "type": "string"},
                        },
                        "required": ["explanation", "output"],
                        "title": "Step",
                        "type": "object",
                        "additionalProperties": False,
                    }
                },
                "properties": {
                    "steps": {"items": {"$ref": "#/$defs/Step"}, "title": "Steps", "type": "array"},
                    "final_answer": {"title": "Final Answer", "type": "string"},
                },
                "required": ["steps", "final_answer"],
                "title": "Solving",
                "type": "object",
                "additionalProperties": False,
            },
            "strict": True,
        }
    },
    input=[
        {
            "role": "user",
            "content": "how can I solve 8x + 7y = -23, and 4x=12? Reply in JSON format.",
        }
    ],
    store=False,
)

In [22]:
json_text = resp_c.output[0].content[0].text
solving = Solving.model_validate_json(json_text)
print(solving)

Solving(
    steps=[
        Step(explanation='Solve the second equation first: 4x = 12, so x = 3.', output='x = 3'),
        Step(explanation='Substitute x = 3 into the first equation: 8(3) + 7y = -23.', output='24 + 7y = -23'),
        Step(explanation='Subtract 24 from both sides: 7y = -47.', output='7y = -47'),
        Step(explanation='Divide both sides by 7: y = -47/7.', output='y = -47/7')
    ],
    final_answer='The solution is x = 3 and y = -47/7.'
)

### 使用responses.parse

In [10]:
resp_p = client.responses.parse(
    model=model_id,
    text_format=Solving,
    input=[
        {
            "role": "user",
            "content": "how can I solve 8x + 7y = -23, and 4x=12? Reply in JSON format.",
        }
    ],
)

In [15]:
print(resp_p.output[0].content[0].text)

{"steps":[{"explanation":"Solve the second equation first: 4x = 12, so divide both sides by 4.","output":"x = 
3"},{"explanation":"Substitute x = 3 into the first equation: 8x + 7y = -23.","output":"8(3) + 7y = 
-23"},{"explanation":"Simplify and solve for y.","output":"24 + 7y = -23  =>  7y = -47  =>  y = 
-47/7"}],"final_answer":"The solution is x = 3 and y = -47/7."}

In [16]:
reasoned_result = Solving.model_validate_json(resp_p.output[0].content[0].text)

In [17]:
reasoned_result

Solving(steps=[Step(explanation='Solve the second equation first: 4x = 12, so divide both sides by 4.', output='x = 3'), Step(explanation='Substitute x = 3 into the first equation: 8x + 7y = -23.', output='8(3) + 7y = -23'), Step(explanation='Simplify and solve for y.', output='24 + 7y = -23  =>  7y = -47  =>  y = -47/7')], final_answer='The solution is x = 3 and y = -47/7.')